# 🎓 Mini-Projet ML : Prédiction de la Performance des Étudiants

**Objectif :** Mettre en place un workflow Machine Learning complet (EDA, nettoyage, modélisation, optimisation) et orchestrer le tout avec Apache Airflow.

**Dataset :** Student Performance Dataset (Note finale G3).

**Environnement :** Python 3.14.7

## 📦 1. Imports et Configuration

Nous commençons par importer toutes les bibliothèques nécessaires. 
- `pandas` et `numpy` pour la manipulation des données.
- `matplotlib` et `seaborn` pour la visualisation (EDA).
- `scikit-learn` pour le Machine Learning (prétraitement, modèles, évaluation).
- `joblib` pour sauvegarder les modèles.

In [ ]:
import sys
import io
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, mean_absolute_error, 
                             mean_squared_error, r2_score)

import joblib

# Configuration
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid')

print(f'✅ Environnement initialisé. Version Python : {sys.version.split()[0]}')

## 📊 2. Chargement et Nettoyage des Données

Le dataset fourni contient une particularité : les séparateurs sont mixtes (certaines lignes utilisent la virgule `,`, d'autres le point-virgule `;`). 

Pour éviter les erreurs de parsing, nous lisons le fichier en texte brut, unifions tous les séparateurs en remplaçant les `;` par des `,`, puis nous chargeons le tout dans un DataFrame Pandas. Nous vérifions ensuite les doublons et les types.

In [ ]:
file_path = 'student-mat.csv'

# Lecture du fichier brut et correction des séparateurs mixtes
with open(file_path, 'r', encoding='utf-8') as f:
    raw_data = f.read()

# Remplacement des points-virgules par des virgules
raw_data = raw_data.replace(';', ',')

# Chargement dans Pandas
df = pd.read_csv(io.StringIO(raw_data))

# Nettoyage basique
df = df.drop_duplicates() # Suppression des doublons

# Conversion de G1 et G2 en numérique (au cas où ils seraient lus comme strings)
df['G1'] = pd.to_numeric(df['G1'], errors='coerce')
df['G2'] = pd.to_numeric(df['G2'], errors='coerce')

print(f'📏 Dimensions du dataset : {df.shape}')
print('\n🔍 Aperçu des 5 premières lignes :')
display(df.head())

print(f'\n❓ Nombre total de valeurs manquantes : {df.isnull().sum().sum()}')

## 🔍 3. Exploration des Données (EDA)

Avant de modéliser, il est crucial de comprendre la distribution de notre cible (G3) et les corrélations entre les variables numériques. 
- L'histogramme nous montre la répartition des notes finales.
- La matrice de corrélation nous aide à identifier les variables les plus liées à la note finale (comme G1 et G2).

In [ ]:
# Distribution de la note finale (G3)
plt.figure(figsize=(10, 5))
sns.histplot(df['G3'], bins=20, kde=True, color='skyblue')
plt.title('Distribution de la Note Finale (G3)')
plt.xlabel('Note (0-20)')
plt.ylabel('Fréquence')
plt.show()

# Matrice de corrélation pour les variables numériques
plt.figure(figsize=(12, 8))
corr_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matrice de Corrélation des Variables Numériques')
plt.show()

## 🛠️ 4. Feature Engineering et Séparation des Cibles

**Règle d'or du Machine Learning : Éviter la fuite de données (Data Leakage).**
Les colonnes `G1` et `G2` sont les notes des premier et deuxième trimestres. Si nous voulons prédire la note finale `G3` au *début* de l'année scolaire, nous ne pouvons pas utiliser `G1` et `G2` comme variables prédictives, car elles ne seront pas encore connues.

Nous les supprimons donc de nos features (X).

Ensuite, nous créons deux cibles :
1. **Classification :** `pass_fail` (1 si G3 >= 10, sinon 0).
2. **Régression :** `G3` (valeur continue).

In [ ]:
# Création de la cible binaire pour la classification (Seuil de réussite = 10)
df['pass_fail'] = (df['G3'] >= 10).astype(int)

# Séparation des features (X) et des cibles (y)
# On retire G1, G2 (fuite de données), G3 (cible), et pass_fail (cible)
features_to_drop = ['G1', 'G2', 'G3', 'pass_fail']
X = df.drop(columns=features_to_drop)

y_class = df['pass_fail'] # Cible pour la Classification
y_reg = df['G3']          # Cible pour la Régression

print('🎯 Répartition des classes pour la Classification (Pass/Fail) :')
print(y_class.value_counts(normalize=True))

## 🏷️ 5. Classification : Prétraitement et Baseline

Pour la classification, nous devons gérer les variables catégorielles (OneHotEncoder) et numériques (StandardScaler).

Nous utilisons un `ColumnTransformer` et un `Pipeline` pour encapsuler le prétraitement. Cela garantit que les transformations sont appliquées correctement aux données de test sans fuite d'information.

Enfin, nous entraînons un **Baseline** (`DummyClassifier`). C'est un modèle "naïf" qui se contente de prédire la classe majoritaire. Il nous sert de point de comparaison : si notre modèle complexe ne bat pas le baseline, il est inutile.

In [ ]:
# Identification des types de colonnes
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(include=['number']).columns

# Pipeline de prétraitement
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

# Split Train/Test (Stratifié pour garder les proportions des classes)
X_train, X_test, y_train_class, y_test_class = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# 1. Baseline
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=42)
baseline_clf.fit(X_train, y_train_class)
y_pred_base = baseline_clf.predict(X_test)

print(f'Accuracy du Baseline : {accuracy_score(y_test_class, y_pred_base):.4f}')

## 🤖 6. Classification : Modèles et Évaluation

Nous testons deux modèles classiques :
1. **Régression Logistique :** Un modèle linéaire, rapide et interprétable.
2. **Random Forest :** Un modèle basé sur des arbres de décision, capable de capturer des relations non-linéaires complexes.

Nous évaluons les modèles avec les métriques standards : Accuracy, Precision, Recall, F1-score, et la Matrice de Confusion.

In [ ]:
# Définition des modèles avec Pipeline
lr_clf = Pipeline(steps=[('preprocessor', preprocessor),
                         ('classifier', LogisticRegression(max_iter=1000, random_state=42))])

rf_clf = Pipeline(steps=[('preprocessor', preprocessor),
                         ('classifier', RandomForestClassifier(random_state=42))])

# Entraînement
lr_clf.fit(X_train, y_train_class)
rf_clf.fit(X_train, y_train_class)

# Prédictions
y_pred_lr = lr_clf.predict(X_test)
y_pred_rf = rf_clf.predict(X_test)

# Fonction d'évaluation
def evaluate_classification(y_true, y_pred, model_name):
    print(f'\n--- 📊 {model_name} ---')
    print(f'Accuracy  : {accuracy_score(y_true, y_pred):.4f}')
    print(f'Precision : {precision_score(y_true, y_pred, zero_division=0):.4f}')
    print(f'Recall    : {recall_score(y_true, y_pred, zero_division=0):.4f}')
    print(f'F1-score  : {f1_score(y_true, y_pred, zero_division=0):.4f}')
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Matrice de Confusion - {model_name}')
    plt.xlabel('Prédit')
    plt.ylabel('Réel')
    plt.show()

# Évaluation
evaluate_classification(y_test_class, y_pred_base, 'Baseline')
evaluate_classification(y_test_class, y_pred_lr, 'Régression Logistique')
evaluate_classification(y_test_class, y_pred_rf, 'Random Forest')

## 🚀 7. Classification : Optimisation des Hyperparamètres

Le Random Forest a montré de bons résultats. Nous allons maintenant chercher la meilleure combinaison d'hyperparamètres (nombre d'arbres, profondeur max, etc.) en utilisant `GridSearchCV`. 

Cette technique teste toutes les combinaisons définies dans `param_grid` via une validation croisée (cv=3) et retient la meilleure configuration selon le F1-score.

In [ ]:
# Grille d'hyperparamètres pour le Random Forest
param_grid_clf = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

# GridSearchCV
grid_search_clf = GridSearchCV(rf_clf, param_grid_clf, cv=3, scoring='f1', n_jobs=-1)
grid_search_clf.fit(X_train, y_train_class)

print(f'🏆 Meilleurs paramètres : {grid_search_clf.best_params_}')

# Évaluation du meilleur modèle
best_rf_clf = grid_search_clf.best_estimator_
y_pred_best_rf = best_rf_clf.predict(X_test)

evaluate_classification(y_test_class, y_pred_best_rf, 'Random Forest Optimisé')

## 📈 8. Régression : Modélisation et Évaluation

Passons maintenant à la régression. L'objectif est de prédire la note exacte `G3` (valeur continue).

Nous comparons :
1. **Baseline :** Prédit simplement la moyenne des notes.
2. **Régression Linéaire :** Modèle linéaire de base.
3. **Random Forest Regressor :** Modèle non-linéaire.

Les métriques utilisées sont le MAE (erreur absolue), le RMSE (erreur quadratique) et le R² (variance expliquée).

In [ ]:
# Split Train/Test pour la régression
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

# 1. Baseline
baseline_reg = DummyRegressor(strategy='mean')
baseline_reg.fit(X_train_reg, y_train_reg)
y_pred_base_reg = baseline_reg.predict(X_test_reg)

# 2. Régression Linéaire
lr_reg = Pipeline(steps=[('preprocessor', preprocessor),
                         ('regressor', LinearRegression())])
lr_reg.fit(X_train_reg, y_train_reg)
y_pred_lr_reg = lr_reg.predict(X_test_reg)

# 3. Random Forest Regressor
rf_reg = Pipeline(steps=[('preprocessor', preprocessor),
                         ('regressor', RandomForestRegressor(random_state=42))])
rf_reg.fit(X_train_reg, y_train_reg)
y_pred_rf_reg = rf_reg.predict(X_test_reg)

# Fonction d'évaluation pour la régression
def evaluate_regression(y_true, y_pred, model_name):
    print(f'\n--- 📈 {model_name} ---')
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'MAE  : {mae:.4f}')
    print(f'RMSE : {rmse:.4f}')
    print(f'R²   : {r2:.4f}')

# Évaluation
evaluate_regression(y_test_reg, y_pred_base_reg, 'Baseline')
evaluate_regression(y_test_reg, y_pred_lr_reg, 'Régression Linéaire')
evaluate_regression(y_test_reg, y_pred_rf_reg, 'Random Forest')

## 🎯 9. Régression : Optimisation

Comme pour la classification, le Random Forest Regressor surpasse la régression linéaire. Nous optimisons ses hyperparamètres avec `GridSearchCV` en utilisant le `neg_mean_squared_error` comme critère de sélection.

In [ ]:
# Grille d'hyperparamètres pour le Random Forest Regressor
param_grid_reg = {
    'regressor__n_estimators': [50, 100, 200],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5]
}

# GridSearchCV
grid_search_reg = GridSearchCV(rf_reg, param_grid_reg, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search_reg.fit(X_train_reg, y_train_reg)

print(f'🏆 Meilleurs paramètres : {grid_search_reg.best_params_}')

# Évaluation du meilleur modèle
best_rf_reg = grid_search_reg.best_estimator_
y_pred_best_rf_reg = best_rf_reg.predict(X_test_reg)

evaluate_regression(y_test_reg, y_pred_best_rf_reg, 'Random Forest Optimisé')

# Sauvegarde du meilleur modèle de classification pour la suite
joblib.dump(best_rf_clf, 'best_classification_model.pkl')
joblib.dump(best_rf_reg, 'best_regression_model.pkl')
print('\n💾 Modèles sauvegardés avec succès (joblib).')

## 🌬️ 10. Orchestration avec Apache Airflow

Pour industrialiser ce workflow, nous utilisons Apache Airflow. 

Le code ci-dessous définit un DAG (Directed Acyclic Graph) nommé `student_performance_ml_pipeline`. 

**Note pour l'exécution :** Ce code est écrit en Python standard. Dans un environnement réel, ce bloc doit être copié dans un fichier `.py` (ex: `student_dag.py`) et placé dans le dossier `dags/` de votre instance Airflow pour que le scheduler puisse l'exécuter. Il ne peut pas être exécuté directement dans ce notebook car Airflow nécessite son propre moteur d'orchestration.

In [ ]:
# ==============================================================================# CODE À COPIER DANS UN FICHIER .py DANS LE DOSSIER DAGS D'AIRFLOW# ==============================================================================
airflow_dag_code = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime
import pandas as pd
import joblib
import io
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Définition des tâches (Tasks)
def extract_and_clean():
    with open('/opt/airflow/data/student-mat.csv', 'r', encoding='utf-8') as f:
        raw_data = f.read().replace(';', ',')
    df = pd.read_csv(io.StringIO(raw_data))
    df = df.drop_duplicates()
    df['pass_fail'] = (df['G3'] >= 10).astype(int)
    df.to_csv('/tmp/cleaned_data.csv', index=False)

def train_model():
    df = pd.read_csv('/tmp/cleaned_data.csv')
    X = df.drop(columns=['G1', 'G2', 'G3', 'pass_fail'])
    y = df['pass_fail']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    preprocessor = ColumnTransformer(transformers=[
        ('num', StandardScaler(), X.select_dtypes(include=['number']).columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), X.select_dtypes(include=['object']).columns)
    ])
    model = Pipeline(steps=[('preprocessor', preprocessor),
                            ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])
    model.fit(X_train, y_train)
    joblib.dump(model, '/tmp/best_model.pkl')

def evaluate_model():
    model = joblib.load('/tmp/best_model.pkl')
    print('✅ Modèle évalué avec succès dans Airflow.')

def save_model():
    print('💾 Modèle sauvegardé dans le stockage persistant.')

# 2. Définition du DAG
default_args = {'owner': 'airflow', 'start_date': datetime(2023, 1, 1), 'retries': 1}

with DAG('student_performance_ml_pipeline', default_args=default_args, schedule_interval=None, catchup=False) as dag:
    
    # 3. Opérateurs
    task1 = PythonOperator(task_id='extract_and_clean', python_callable=extract_and_clean)
    task2 = PythonOperator(task_id='train_model', python_callable=train_model)
    task3 = PythonOperator(task_id='evaluate_model', python_callable=evaluate_model)
    task4 = PythonOperator(task_id='save_model', python_callable=save_model)
    
    # 4. Dépendances (Orchestration linéaire)
    task1 >> task2 >> task3 >> task4
'''

print('✅ Code du DAG Airflow généré avec succès.')
print('Copiez le contenu de la variable airflow_dag_code dans votre fichier .py pour Airflow.')